# Notebook de Análise
Bem-vindo!

In [51]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import datasets

In [52]:
url = "/Users/Ccesconetto/Downloads/Analise1/data/raw/vendas_ecommerce_sujo.csv"
data = pd.read_csv(url)
df = pd.DataFrame(data)
df_clean = df[["id_pedido"]].copy()

In [53]:
df.info()
df.describe()
df.dtypes
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 315 entries, 0 to 314
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id_pedido         315 non-null    int64  
 1   data_pedido       315 non-null    str    
 2   categoria         315 non-null    str    
 3   estado            299 non-null    str    
 4   quantidade        306 non-null    float64
 5   preco_unitario    315 non-null    str    
 6   metodo_pagamento  303 non-null    str    
 7   status_pedido     305 non-null    str    
 8   avaliacao         271 non-null    float64
dtypes: float64(2), int64(1), str(6)
memory usage: 22.3 KB


Index(['id_pedido', 'data_pedido', 'categoria', 'estado', 'quantidade',
       'preco_unitario', 'metodo_pagamento', 'status_pedido', 'avaliacao'],
      dtype='str')

In [54]:
df["data_pedido"].isnull().sum()
# 95 NaN values before the cleaning process;

df_clean["data_pedido_clean"] = pd.to_datetime(
    df["data_pedido"], 
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

# Comandos:
- .isnull().sum() → Conta o total de valores nulos em cada coluna
- pd.to_datetime() → Converte string → datetime64
- format="mixed" → Detecta múltiplos formatos de data automaticamente
- errors="coerce" → Transforma valores inválidos em NaN (ou NaT para datas)

In [55]:
df["preco_unitario"]
df["preco_unitario"].isnull().sum()
# 0 NaN values before the cleaning process;

def normalizar_preco(valor: str) -> str:
    valor = str(valor).strip().replace("R$", "").strip()  
    
    if "," in valor:
        valor = valor.replace(".", "").replace(",", ".")  
    
    return valor
    return valor    

df_clean["preco_unitario_clean"] = (
    df["preco_unitario"]
    .apply(normalizar_preco)
    .pipe(pd.to_numeric, errors="coerce")
    .astype(float)
)


# Função normalizar_preco

## Definição
- `def normalizar_preco(valor: str) -> str` → define a função, recebe e retorna `str`

## Limpeza inicial
- `str(valor)` → converte para string, protege contra `float` e `NaN`
- `.strip()` → remove espaços nas bordas
- `.replace("R$", "")` → remove o prefixo de moeda
- `.strip()` → remove o espaço que sobrou após o `R$`

## Conversão de formato
- `if "," in valor` → identifica formato pt-BR (`"507,87"`) vs neutro (`"110.25"`)
- `.replace(".", "")` → remove separador de milhar: `"1.235,50"` → `"1235,50"`
- `.replace(",", ".")` → troca vírgula decimal por ponto: `"1235,50"` → `"1235.50"`
- `return valor` → devolve a string normalizada

## Pipeline
- `.apply(normalizar_preco)` → executa a função linha por linha na coluna
- `.pipe(pd.to_numeric, errors="coerce")` → converte string → número; inválidos → NaN
- `.astype(float)` → garante tipo final `float64`

In [56]:
df["categoria"].dtypes
df["categoria"].unique()
df["categoria"].value_counts()
df_clean["categoria_clean"] = df["categoria"].str.title()


# Comandos:
- df[""].unique() → Retorna todos os valores únicos da coluna
- df[""].value_counts() → Contagem de ocorrências de cada valor único
- df[""].str.title() → Primeira letra maiúscula de cada palavra do valor

In [57]:
df_clean["metodo_pagamento_clean"] = (
    df["metodo_pagamento"]
    .str.strip()
    .str.title()
)

In [58]:
mapa_estados = {
    "São Paulo"         : "SP", "SAO PAULO"       : "SP",
    "Sao Paulo"         : "SP", "S.P."            : "SP",
    "sp"                : "SP",
    "Rio de Janeiro"    : "RJ", "RIO DE JANEIRO"  : "RJ",
    "R.J."              : "RJ", "rj"              : "RJ",
    "Minas Gerais"      : "MG", "MINAS GERAIS"    : "MG",
    "Minas gerais"      : "MG", "mg"              : "MG",
    "Rio Grande do Sul" : "RS", "RIO GRANDE DO SUL": "RS",
    "rs"                : "RS",
    "Bahia"             : "BA", "BAHIA"           : "BA",
    "ba"                : "BA",
}

df_clean["estado_clean"] = (
    df["estado"]
    .str.strip()                       
    .replace(mapa_estados)    
)

In [59]:
df_clean[ "quantidade_clean" ]= df[ "quantidade" ]

In [60]:
df["status_pedido"].unique()
df["status_pedido"].value_counts()
df_clean["status_clean"] = df["status_pedido"].fillna("Desconhecido").str.title()

In [61]:
df.columns
df.dtypes

id_pedido             int64
data_pedido             str
categoria               str
estado                  str
quantidade          float64
preco_unitario          str
metodo_pagamento        str
status_pedido           str
avaliacao           float64
dtype: object

In [62]:
df_clean["avaliacao_clean"] = (
    df["avaliacao"]
    .replace(".", "")
    .astype("Int64")  
)

df_clean["avaliacao_clean"].unique()

<IntegerArray>
[<NA>, 5, 1, 4, 3, 2]
Length: 6, dtype: Int64

In [63]:
df_clean.info()
df_clean.describe()
df_clean.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 315 entries, 0 to 314
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   id_pedido               315 non-null    int64         
 1   data_pedido_clean       315 non-null    datetime64[us]
 2   preco_unitario_clean    315 non-null    float64       
 3   categoria_clean         315 non-null    str           
 4   metodo_pagamento_clean  303 non-null    str           
 5   estado_clean            299 non-null    str           
 6   quantidade_clean        306 non-null    float64       
 7   status_clean            315 non-null    str           
 8   avaliacao_clean         271 non-null    Int64         
dtypes: Int64(1), datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 22.6 KB


id_pedido                  0
data_pedido_clean          0
preco_unitario_clean       0
categoria_clean            0
metodo_pagamento_clean    12
estado_clean              16
quantidade_clean           9
status_clean               0
avaliacao_clean           44
dtype: int64

In [64]:
df_clean = df_clean.drop_duplicates(subset=["id_pedido"], keep="first")
df_clean = df_clean[df_clean["preco_unitario_clean"] > 0]
df_clean = df_clean[df_clean["quantidade_clean"] > 0]
df_clean.info()
df_clean.describe()

<class 'pandas.DataFrame'>
Index: 285 entries, 0 to 314
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   id_pedido               285 non-null    int64         
 1   data_pedido_clean       285 non-null    datetime64[us]
 2   preco_unitario_clean    285 non-null    float64       
 3   categoria_clean         285 non-null    str           
 4   metodo_pagamento_clean  273 non-null    str           
 5   estado_clean            271 non-null    str           
 6   quantidade_clean        285 non-null    float64       
 7   status_clean            285 non-null    str           
 8   avaliacao_clean         246 non-null    Int64         
dtypes: Int64(1), datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 22.5 KB


,id_pedido,data_pedido_clean,preco_unitario_clean,quantidade_clean,avaliacao_clean
count,285.000000,285,285.000000,285.000000,246.0
mean,1148.768421,2023-12-23 15:39:47.368421,772.111754,8.431579,3.800813
min,1001.000000,2023-01-01 00:00:00,20.400000,1.000000,1.0
25%,1075.000000,2023-06-23 00:00:00,218.980000,3.000000,3.0
50%,1147.000000,2023-12-23 00:00:00,410.600000,5.000000,4.0
75%,1222.000000,2024-06-23 00:00:00,624.210000,7.000000,5.0
max,1300.000000,2024-12-31 00:00:00,99999.000000,500.000000,5.0
std,85.323839,NaN,5902.742157,41.478143,1.134325


In [65]:
df_clean.to_csv("/Users/Ccesconetto/Downloads/Analise1/data/processed/vendas_ecommerce_limpo.csv", index=False) 